# Calibration based on entropy scores from attention and final layers: 
1) Best head selection in terms of HDP or ROC AUC.
2) Gathering entropy scores from best attn heads for answer token into feature vector.
3) Getting calibrated confidence based on MLP-only, MLP+Beta or Beta-only calibration heads.

More information in docs section TODO

## Environment preparation

In [1]:
import os
import torch
from functools import partial
from tqdm import tqdm

from services.index import Index, IndexDataset
from services.experiment.cropped.data_process_utils import process_elements_hal, process_elements_main
from services.common.calibration_heads import (
    MLPBetaCalibrationHead,
    MLPCalibrationHead,
    WeightedBetaCalibrationHead,
)
from services.experiment.calibration_utils import (
    find_best_layer_head_hal_dif_power,
    find_best_layer_head_roc_auc,
    fit_hparameters,
    test_calibration_model
)


In [2]:
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '3')
print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}")

CUDA_VISIBLE_DEVICES=3


In [3]:
torch.random.manual_seed(42)

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"Index {i}: {props.name}, UUID: {props.uuid}")
    
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.set_device(device)
print(f"Chosen device: {device}")

Index 0: NVIDIA L40, UUID: 7b7e27a5-a434-d736-eabd-b9414645af76
Chosen device: cuda:0


In [4]:
LAYERS_COUNT = 28
HEADS_COUNT = 28 
HEDAS_GROUP_SIZES = [1, 3, 5, 7, 10, 15, 20, 30]
HPARAMETER_SEARCH_TRIALS = 20
BEST_HEADS_GROUP_SIZE = 30
FEATURES_COUNT = 0
ATTN_ONLY = True
HS_DATA_SIZE = 50
L1_REG = True
L2_REG = False

LOGS_ROOT_DIR = "./logs_l1_hs_50/"
LOGS_MODE_DIR = "attn_only" if ATTN_ONLY else "attn_plus_final"
LOGS_DIR = f"{LOGS_ROOT_DIR}{LOGS_MODE_DIR}/"

## Data preparation 

In [ ]:
index = Index("../../../../../index_data/qwen2.5_MMLU-PRO_attn_cropped_12000")

## Head selection

In [6]:
head_selection_dataset = IndexDataset(
    index=index, 
    process_elements=partial(
        process_elements_hal,
        layers_count=LAYERS_COUNT,
        heads_count=HEADS_COUNT,
        verbose=True,
        device=device
    ),
    split="val",
    load_all_data=True,
    verbose=True
)

head_selection_data = head_selection_dataset.get(HS_DATA_SIZE) if HS_DATA_SIZE > 0 \
     else head_selection_dataset.get()

Processing data...: 100%|██████████| 28/28 [00:00<00:00, 2970.62it/s]


In [7]:
hal_dif_best_layers, hal_dif_best_heads = find_best_layer_head_hal_dif_power(
    data=head_selection_data,
    layers_count=LAYERS_COUNT,
    heads_count=HEADS_COUNT,
    best_heads_group_size=BEST_HEADS_GROUP_SIZE
)
print(f"Best Layers and Heads: {[(x[0].item(), x[1].item()) for x in zip(hal_dif_best_layers, hal_dif_best_heads)]}")

Best Layers and Heads: [(9, 26), (13, 16), (14, 16), (16, 17), (16, 1), (7, 14), (16, 20), (16, 5), (13, 23), (14, 15), (9, 17), (0, 15), (8, 26), (12, 26), (13, 12), (16, 0), (9, 13), (19, 17), (10, 11), (14, 11), (15, 1), (16, 2), (23, 17), (7, 23), (9, 9), (14, 22), (15, 16), (15, 24), (18, 13), (20, 14)]


In [8]:
roc_auc_best_layers, roc_auc_best_heads = find_best_layer_head_roc_auc(
    data=head_selection_data,
    layers_count=LAYERS_COUNT,
    heads_count=HEADS_COUNT,
    best_heads_group_size=BEST_HEADS_GROUP_SIZE
)
print(f"Best Layers and Heads: {[(x[0].item(), x[1].item()) for x  in zip(roc_auc_best_layers, roc_auc_best_heads)]}")

Best Layers and Heads: [(13, 16), (16, 5), (14, 16), (13, 3), (14, 11), (16, 2), (13, 21), (23, 10), (14, 26), (14, 22), (4, 7), (13, 23), (13, 12), (16, 20), (16, 26), (7, 14), (9, 9), (20, 14), (12, 26), (10, 11), (14, 15), (14, 23), (16, 1), (15, 24), (14, 10), (8, 23), (9, 26), (7, 15), (13, 15), (13, 11)]


## Splits preparing

In [9]:
hal_train = IndexDataset(
    index, 
    partial(
        process_elements_main,
        best_layers=hal_dif_best_layers,
        best_heads=hal_dif_best_heads,
        attn_only=ATTN_ONLY,
        device=device,
    ),
    split="train",
    load_all_data=True,
    verbose=True
)
hal_val = IndexDataset(
    index, 
    partial(
        process_elements_main,
        best_layers=hal_dif_best_layers,
        best_heads=hal_dif_best_heads,
        attn_only=ATTN_ONLY,
        device=device,
    ),
    split="val",
    load_all_data=True,
    verbose=True
)
hal_test = IndexDataset(
    index, 
    partial(
        process_elements_main,
        best_layers=hal_dif_best_layers,
        best_heads=hal_dif_best_heads,
        attn_only=ATTN_ONLY,
        device=device,
    ),
    split="test",
    load_all_data=True,
    verbose=True
)

Processing data...: 100%|██████████| 1199/1199 [00:00<00:00, 3152.49it/s]


In [10]:
rocauc_train = IndexDataset(
    index, 
    partial(
        process_elements_main,
        best_layers=roc_auc_best_layers,
        best_heads=roc_auc_best_heads,
        attn_only=ATTN_ONLY,
        device=device,
    ),
    split="train",
    load_all_data=True,
    verbose=True
)
rocauc_val = IndexDataset(
    index, 
    partial(
        process_elements_main,
        best_layers=roc_auc_best_layers,
        best_heads=roc_auc_best_heads,
        attn_only=ATTN_ONLY,
        device=device,
    ),    
    split="val",
    load_all_data=True,
    verbose=True
)
rocauc_test = IndexDataset(
    index, 
    partial(
        process_elements_main,
        best_layers=roc_auc_best_layers,
        best_heads=roc_auc_best_heads,
        attn_only=ATTN_ONLY,
        device=device,
    ),    
    split="test",
    load_all_data=True,
    verbose=True
)

Processing data...: 100%|██████████| 1199/1199 [00:01<00:00, 678.03it/s]


## Calibration with HDP head selection data

### MLP-only calibration

In [11]:

local_logs_dir = LOGS_DIR + "(mlp)(hal)calibration_res/"

hal_test_data = hal_test.get()
for heads_count in tqdm(HEDAS_GROUP_SIZES, desc="Calibrating with various heads"):
    feature_ids = torch.cat([
        torch.arange(
            k * (BEST_HEADS_GROUP_SIZE + (not ATTN_ONLY)),
            k * (BEST_HEADS_GROUP_SIZE + (not ATTN_ONLY)) + (heads_count + (not ATTN_ONLY)),
            dtype=torch.long
        )
        for k in range(FEATURES_COUNT + 1)
    ])
    
    fit_results = fit_hparameters(
        model_class=MLPCalibrationHead,
        train=hal_train,
        test=hal_val,
        attn_only=ATTN_ONLY,
        l1_reg=L1_REG,
        l2_reg=L2_REG,
        features_count=FEATURES_COUNT,
        feature_ids=feature_ids,
        heads_count=heads_count,
        device=device,
        logging=True,
        log_dir=local_logs_dir + f"train#{heads_count}"
    )
    
    model = MLPCalibrationHead(
        in_features=(FEATURES_COUNT + 1) * (heads_count + (not ATTN_ONLY)),
        device=device
    )
    model.load_state_dict(fit_results["parameters"])
    model.eval()
    
    test_calibrated_probs = model.calibrate(hal_test_data["features"][:, feature_ids], device)
    test_calibration_model(
        test_calibrated_probs.to(device=device, dtype=torch.float32),
        hal_test_data["labels"].to(device=device, dtype=torch.float32),
        device=device,
        logging=True,
        log_dir=local_logs_dir + f"test#{heads_count}",
    )

Calibrating with various heads: 100%|██████████| 8/8 [17:57<00:00, 134.71s/it]


### MLP+Beta calibration

In [12]:

local_logs_dir = LOGS_DIR + "(mlp+beta)(hal)calibration_res/"

hal_test_data = hal_test.get()
for heads_count in tqdm(HEDAS_GROUP_SIZES, desc="Calibrating with various heads"):
    feature_ids = torch.cat([
        torch.arange(
            k * (BEST_HEADS_GROUP_SIZE + (not ATTN_ONLY)),
            k * (BEST_HEADS_GROUP_SIZE + (not ATTN_ONLY)) + (heads_count + (not ATTN_ONLY)),
            dtype=torch.long
        )
        for k in range(FEATURES_COUNT + 1)
    ])
    
    fit_results = fit_hparameters(
        model_class=MLPBetaCalibrationHead,
        train=hal_train,
        test=hal_val,
        attn_only=ATTN_ONLY,
        l1_reg=L1_REG,
        l2_reg=L2_REG,
        features_count=FEATURES_COUNT,
        feature_ids=feature_ids,
        heads_count=heads_count,
        device=device,
        logging=True,
        log_dir=local_logs_dir + f"train#{heads_count}"
    )
    
    model = MLPBetaCalibrationHead(
        in_features=(FEATURES_COUNT + 1) * (heads_count + (not ATTN_ONLY)),
        device=device
    )
    model.load_state_dict(fit_results["parameters"])
    model.eval()
    
    test_calibrated_probs = model.calibrate(hal_test_data["features"][:, feature_ids], device)
    test_calibration_model(
        test_calibrated_probs.to(device=device, dtype=torch.float32),
        hal_test_data["labels"].to(device=device, dtype=torch.float32),
        device=device,
        logging=True,
        log_dir=local_logs_dir + f"test#{heads_count}",
    )

Calibrating with various heads: 100%|██████████| 8/8 [38:36<00:00, 289.51s/it]


### Weighted Beta calibration

In [13]:
local_logs_dir = LOGS_DIR + "(weighted_beta)(hal)calibration_res/"

hal_test_data = hal_test.get()
for heads_count in tqdm(HEDAS_GROUP_SIZES, desc="Calibrating with various heads"):
    feature_ids = torch.cat([
        torch.arange(
            k * (BEST_HEADS_GROUP_SIZE + (not ATTN_ONLY)),
            k * (BEST_HEADS_GROUP_SIZE + (not ATTN_ONLY)) + (heads_count + (not ATTN_ONLY)),
            dtype=torch.long
        )
        for k in range(FEATURES_COUNT + 1)
    ])
    
    fit_results = fit_hparameters(
        model_class=WeightedBetaCalibrationHead,
        train=hal_train,
        test=hal_val,
        attn_only=ATTN_ONLY,
        l1_reg=L1_REG,
        l2_reg=L2_REG,
        features_count=FEATURES_COUNT,
        feature_ids=feature_ids,
        heads_count=heads_count,
        device=device,
        logging=True,
        log_dir=local_logs_dir + f"train#{heads_count}"
    )
    
    model = WeightedBetaCalibrationHead(
        in_features=(FEATURES_COUNT + 1) * (heads_count + (not ATTN_ONLY)),
        device=device
    )
    model.load_state_dict(fit_results["parameters"])
    model.eval()
    
    test_calibrated_probs = model.calibrate(hal_test_data["features"][:, feature_ids], device)
    test_calibration_model(
        test_calibrated_probs.to(device=device, dtype=torch.float32),
        hal_test_data["labels"].to(device=device, dtype=torch.float32),
        device=device,
        logging=True,
        log_dir=local_logs_dir + f"test#{heads_count}",
    )

Calibrating with various heads: 100%|██████████| 8/8 [36:21<00:00, 272.63s/it]


## Calibration with ROC-AUC head selection data

### MLP-only calibration

In [14]:

local_logs_dir = LOGS_DIR + "(mlp)(roc-auc)calibration_res/"

rocauc_test_data = rocauc_test.get()
for heads_count in tqdm(HEDAS_GROUP_SIZES, desc="Calibrating with various heads"):
    feature_ids = torch.cat([
        torch.arange(
            k * (BEST_HEADS_GROUP_SIZE + (not ATTN_ONLY)),
            k * (BEST_HEADS_GROUP_SIZE + (not ATTN_ONLY)) + (heads_count + (not ATTN_ONLY)),
            dtype=torch.long
        )
        for k in range(FEATURES_COUNT + 1)
    ])
    
    fit_results = fit_hparameters(
        model_class=MLPCalibrationHead,
        train=rocauc_train,
        test=rocauc_val,
        attn_only=ATTN_ONLY,
        l1_reg=L1_REG,
        l2_reg=L2_REG,
        features_count=FEATURES_COUNT,
        feature_ids=feature_ids,
        heads_count=heads_count,
        device=device,
        logging=True,
        log_dir=local_logs_dir + f"train#{heads_count}"
    )
    
    model = MLPCalibrationHead(
        in_features=(FEATURES_COUNT + 1) * (heads_count + (not ATTN_ONLY)),
        device=device
    )
    model.load_state_dict(fit_results["parameters"])
    model.eval()
    
    test_calibrated_probs = model.calibrate(rocauc_test_data["features"][:, feature_ids], device)
    test_calibration_model(
        test_calibrated_probs.to(device=device, dtype=torch.float32),
        rocauc_test_data["labels"].to(device=device, dtype=torch.float32),
        device=device,
        logging=True,
        log_dir=local_logs_dir + f"test#{heads_count}",
    )

Calibrating with various heads: 100%|██████████| 8/8 [30:48<00:00, 231.06s/it]


### MLP+Beta calibration

In [15]:

local_logs_dir = LOGS_DIR + "(mlp+beta)(roc-auc)calibration_res/"

rocauc_test_data = rocauc_test.get()
for heads_count in tqdm(HEDAS_GROUP_SIZES, desc="Calibrating with various heads"):
    feature_ids = torch.cat([
        torch.arange(
            k * (BEST_HEADS_GROUP_SIZE + (not ATTN_ONLY)),
            k * (BEST_HEADS_GROUP_SIZE + (not ATTN_ONLY)) + (heads_count + (not ATTN_ONLY)),
            dtype=torch.long
        )
        for k in range(FEATURES_COUNT + 1)
    ])
    
    fit_results = fit_hparameters(
        model_class=MLPBetaCalibrationHead,
        train=rocauc_train,
        test=rocauc_val,
        attn_only=ATTN_ONLY,
        l1_reg=L1_REG,
        l2_reg=L2_REG,
        features_count=FEATURES_COUNT,
        feature_ids=feature_ids,
        heads_count=heads_count,
        device=device,
        logging=True,
        log_dir=local_logs_dir + f"train#{heads_count}"
    )
    
    model = MLPBetaCalibrationHead(
        in_features=(FEATURES_COUNT + 1) * (heads_count + (not ATTN_ONLY)),
        device=device
    )
    model.load_state_dict(fit_results["parameters"])
    model.eval()
    
    test_calibrated_probs = model.calibrate(rocauc_test_data["features"][:, feature_ids], device)
    test_calibration_model(
        test_calibrated_probs.to(device=device, dtype=torch.float32),
        rocauc_test_data["labels"].to(device=device, dtype=torch.float32),
        device=device,
        logging=True,
        log_dir=local_logs_dir + f"test#{heads_count}",
    )

Calibrating with various heads: 100%|██████████| 8/8 [34:44<00:00, 260.61s/it]


### BetaW calibration

In [16]:
local_logs_dir = LOGS_DIR + "(weighted_beta)(roc-auc)calibration_res/"

rocauc_test_data = rocauc_test.get()
for heads_count in tqdm(HEDAS_GROUP_SIZES, desc="Calibrating with various heads"):
    feature_ids = torch.cat([
        torch.arange(
            k * (BEST_HEADS_GROUP_SIZE + (not ATTN_ONLY)),
            k * (BEST_HEADS_GROUP_SIZE + (not ATTN_ONLY)) + (heads_count + (not ATTN_ONLY)),
            dtype=torch.long
        )
        for k in range(FEATURES_COUNT + 1)
    ])
    
    fit_results = fit_hparameters(
        model_class=WeightedBetaCalibrationHead,
        train=rocauc_train,
        test=rocauc_val,
        attn_only=ATTN_ONLY,
        l1_reg=L1_REG,
        l2_reg=L2_REG,
        features_count=FEATURES_COUNT,
        feature_ids=feature_ids,
        heads_count=heads_count,
        device=device,
        logging=True,
        log_dir=local_logs_dir + f"train#{heads_count}"
    )
    
    model = WeightedBetaCalibrationHead(
        in_features=(FEATURES_COUNT + 1) * (heads_count + (not ATTN_ONLY)),
        device=device
    )
    model.load_state_dict(fit_results["parameters"])
    model.eval()
    
    test_calibrated_probs = model.calibrate(rocauc_test_data["features"][:, feature_ids], device)
    test_calibration_model(
        test_calibrated_probs.to(device=device, dtype=torch.float32),
        rocauc_test_data["labels"].to(device=device, dtype=torch.float32),
        device=device,
        logging=True,
        log_dir=local_logs_dir + f"test#{heads_count}",
    )

Calibrating with various heads: 100%|██████████| 8/8 [34:01<00:00, 255.20s/it]
